In [1]:
!pip install -q uv
!uv pip install --system tensorflow-data-validation

Using Python 3.11.13 environment at: /usr
Checked 1 package in 300ms


In [2]:
#
import pandas as pd
import numpy as np
import tensorflow_data_validation as tfdv

In [4]:
df_acc = pd.read_csv('moneycraft_accounts.csv')
df_acc['month'] = pd.PeriodIndex(df_acc['month'], freq='M')
df_acc.head()

,customer_id,month,acquisition_channel,region,age_band,account_tier,monthly_fee,joined_on_promo
0,1,2022-01,Comparison Site,Scotland,25-34,Classic,0,False
1,2,2022-01,Online,London,55-64,Silver,11,False
2,3,2022-01,Branch,South East,25-34,Platinum,28,False
3,4,2022-01,Referral,London,25-34,Silver,0,True
4,5,2022-01,Comparison Site,South East,35-44,Silver,11,False


In [5]:
df_mark = pd.read_csv('moneycraft_marketing_spend.csv')
df_mark['month'] = pd.PeriodIndex(df_mark['month'], freq='M')
df_mark.head()

,month,channel,spend_gbp
0,2022-01,Branch,7310.0
1,2022-01,Broker/IFA,4014.0
2,2022-01,Comparison Site,3162.0
3,2022-01,Online,3945.0
4,2022-01,Outbound,4899.0


In [6]:
switches = (df_acc.sort_values(['customer_id', 'month'])
                  .groupby('customer_id')['account_tier']
                  .apply(lambda s: (s != s.shift()).sum() - 1))

switches.head(20)

,account_tier
customer_id,
1,0
2,2
3,2
4,0
5,1
6,2
7,0
8,1
9,2


In [8]:
df_acc['customer_id'].nunique()
#15732

15732

In [9]:
df_acc.dtypes

,0
customer_id,int64
month,period[M]
acquisition_channel,object
region,object
age_band,object
account_tier,object
monthly_fee,int64
joined_on_promo,bool


In [34]:
df_acc = df_acc.sort_values(['customer_id', 'month'])
df_acc

,customer_id,month,acquisition_channel,region,age_band,account_tier,monthly_fee,joined_on_promo
0,1,2022-01,Comparison Site,Scotland,25-34,Classic,0,False
280,1,2022-02,Comparison Site,Scotland,25-34,Classic,0,False
840,1,2022-03,Comparison Site,Scotland,25-34,Classic,0,False
1664,1,2022-04,Comparison Site,Scotland,25-34,Classic,0,False
2750,1,2022-05,Comparison Site,Scotland,25-34,Classic,0,False
...,...,...,...,...,...,...,...,...
179013,15728,2024-12,Outbound,Midlands,35-44,Silver,0,True
179014,15729,2024-12,Comparison Site,Midlands,25-34,Classic,0,True
179015,15730,2024-12,Referral,London,25-34,Gold,18,False
179016,15731,2024-12,Online,South East,55-64,Platinum,28,False


In [18]:
grouped = df_acc.groupby('customer_id')

# A few simple summaries, one per customer:
signup_month  = grouped['month'].min()    # their FIRST month
last_active   = grouped['month'].max()    # their LAST month
tenure_months = grouped['month'].count()  # how many months in total

signup_month.head()

,month
customer_id,
1,2022-01
2,2022-01
3,2022-01
4,2022-01
5,2022-01


In [29]:
grouped['joined_on_promo'].first()

,joined_on_promo
customer_id,
1,False
2,False
3,False
4,True
5,False
...,...
15728,True
15729,True
15730,False


In [41]:
channel    = grouped['acquisition_channel'].first() # Assuming they haven't rejoined using a different channel
region     = grouped['region'].first() # Assuming they haven't moved
age_band   = grouped['age_band'].first() # Assuming they haven't aged
first_plan = grouped['account_tier'].first()   # the plan they STARTED on
last_plan  = grouped['account_tier'].last()    # the plan they ENDED on
promo_join = grouped['joined_on_promo'].first() # whether they joined on promo


first_plan.head()

,account_tier
customer_id,
1,Classic
2,Silver
3,Platinum
4,Silver
5,Silver


In [38]:
# 1. Identify where a tier change occurs
tier_changed = df_acc['account_tier'] != df_acc.groupby('customer_id')['account_tier'].shift()

# 2. Group into a clean Python list per customer
plan_history_list = df_acc[tier_changed].groupby('customer_id')['account_tier'].apply(list)
plan_history_list

,account_tier
customer_id,
1,[Classic]
2,"[Silver, Gold, Platinum]"
3,"[Platinum, Gold, Platinum]"
4,[Silver]
5,"[Silver, Classic]"
...,...
15728,[Silver]
15729,[Classic]
15730,[Gold]


In [40]:
total_revenue = grouped['monthly_fee'].sum()   # everything they ever paid us
max_paid      = grouped['monthly_fee'].max()   # the most they ever paid in a month

total_revenue.head()

,monthly_fee
customer_id,
1,0
2,595
3,848
4,330
5,33


In [42]:
scv = pd.DataFrame({
    'signup_month' : signup_month,
    'last_active'  : last_active,
    'tenure_months': tenure_months,
    'channel'      : channel,
    'region'       : region,
    'age_band'     : age_band,
    'first_plan'   : first_plan,
    'last_plan'    : last_plan,
    'total_revenue': total_revenue,
    'promo_join'   : promo_join,
    'plan_history' : plan_history_list
})

# A handy True/False column: did they EVER actually pay us?
scv['ever_paid'] = max_paid > 0

scv.head()

,signup_month,last_active,tenure_months,channel,region,age_band,first_plan,last_plan,total_revenue,promo_join,plan_history,ever_paid
customer_id,,,,,,,,,,,,
1,2022-01,2022-06,6,Comparison Site,Scotland,25-34,Classic,Classic,0,False,[Classic],False
2,2022-01,2024-12,36,Online,London,55-64,Silver,Platinum,595,False,"[Silver, Gold, Platinum]",True
3,2022-01,2024-12,36,Branch,South East,25-34,Platinum,Platinum,848,False,"[Platinum, Gold, Platinum]",True
4,2022-01,2024-12,36,Referral,London,25-34,Silver,Silver,330,True,[Silver],True
5,2022-01,2022-10,10,Comparison Site,South East,35-44,Silver,Classic,33,False,"[Silver, Classic]",True


In [47]:
# 1. Define the condition for successful promo retention
is_promo = scv['promo_join'] == True
stayed_past_free = scv['tenure_months'] > 6

# 2. Use np.where: True if retained, False if they left early, None/NaN if not a promo customer
scv['retained_after_promo'] = np.where(is_promo, stayed_past_free, None)

In [45]:
final_month = df_acc['month'].max()          # the last month we can see in the data
print('Final month in the data:', final_month)

scv['churned'] = scv['last_active'] < final_month   # True if they left before the end
scv[['signup_month', 'last_active', 'tenure_months', 'churned']].head()

Final month in the data: 2024-12


,signup_month,last_active,tenure_months,churned
customer_id,,,,
1,2022-01,2022-06,6,True
2,2022-01,2024-12,36,False
3,2022-01,2024-12,36,False
4,2022-01,2024-12,36,False
5,2022-01,2022-10,10,True


In [48]:
scv.head()

,signup_month,last_active,tenure_months,channel,region,age_band,first_plan,last_plan,total_revenue,promo_join,plan_history,ever_paid,churned,retained_after_promo
customer_id,,,,,,,,,,,,,,
1,2022-01,2022-06,6,Comparison Site,Scotland,25-34,Classic,Classic,0,False,[Classic],False,True,None
2,2022-01,2024-12,36,Online,London,55-64,Silver,Platinum,595,False,"[Silver, Gold, Platinum]",True,False,None
3,2022-01,2024-12,36,Branch,South East,25-34,Platinum,Platinum,848,False,"[Platinum, Gold, Platinum]",True,False,None
4,2022-01,2024-12,36,Referral,London,25-34,Silver,Silver,330,True,[Silver],True,False,True
5,2022-01,2022-10,10,Comparison Site,South East,35-44,Silver,Classic,33,False,"[Silver, Classic]",True,True,None
